In [1]:
import pandas as pd
import requests
from tqdm import tqdm
from datetime import date

In [2]:
DATES_IN_BATCH = 875 # Results in 4 batches (so the url isn't too long)
URL = "https://ephtracking.cdc.gov/apigateway/api/v1/getCoreHolder/1238/2241/all/all/8/{}/0/0"
RELEVANT_COLUMNS = ["geo", "temporal", "dataValue"]
JSON_TABLE = "regionPMTableResult"

START_DATE = date(2018, 1, 1)
END_DATE = date(2025, 8, 31)

OUTPUT_FILE = "../../data/edv/regional_edv_data.csv"

In [3]:
# all dates between the start and end
dates = pd.date_range(START_DATE, END_DATE).strftime("%Y%m%d")
# batch the dates to get multiple at once
combined_dates = [",".join(dates[i:i + DATES_IN_BATCH]) for i in range(0, len(dates), DATES_IN_BATCH)]

In [4]:
df = pd.DataFrame()
for date_batch in tqdm(combined_dates):
    url = URL.format(date_batch)
    data = requests.get(url).json()
    df_day = pd.DataFrame(data[JSON_TABLE], columns = RELEVANT_COLUMNS)
    df = pd.concat([df, df_day])
df

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:27<00:00,  6.85s/it]


,geo,temporal,dataValue
0,Region 1,20180718,79
1,Region 1,20180719,86
2,Region 1,20180720,60
3,Region 1,20180721,63
4,Region 1,20180722,36
...,...,...,...
1745,Region 10,20250314,0
1746,Region 10,20250313,13
1747,Region 10,20250312,6
1748,Region 10,20250311,6


In [5]:
df["temporal"] = pd.to_datetime(df["temporal"])
df

,geo,temporal,dataValue
0,Region 1,2018-07-18,79
1,Region 1,2018-07-19,86
2,Region 1,2018-07-20,60
3,Region 1,2018-07-21,63
4,Region 1,2018-07-22,36
...,...,...,...
1745,Region 10,2025-03-14,0
1746,Region 10,2025-03-13,13
1747,Region 10,2025-03-12,6
1748,Region 10,2025-03-11,6


In [6]:
df_regional = df.pivot(index = "temporal", columns = "geo", values = "dataValue")[df.geo.unique()]
df_regional.columns.name = None
df_regional

,Region 1,Region 2,Region 3,Region 4,Region 5,Region 6,Region 7,Region 8,Region 9,Region 10
temporal,,,,,,,,,,
2018-01-01,7,11,7,0,9,0,33,0,0,48
2018-01-02,0,13,10,5,8,0,128,0,0,0
2018-01-03,0,6,4,3,0,5,34,0,11,0
2018-01-04,11,5,0,8,6,0,0,0,16,0
2018-01-05,14,4,4,3,11,0,0,0,5,20
...,...,...,...,...,...,...,...,...,...,...
2025-08-27,6,17,15,103,18,234,0,65,260,121
2025-08-28,24,14,24,111,24,297,42,22,207,151
2025-08-29,0,7,25,95,12,204,41,65,233,128


In [7]:
df_regional.to_csv(OUTPUT_FILE)